# Gradio Interface - Medical RAG
## Perbandingan Baseline RAG vs RAG-Fusion pada Dokumen Rekam Medis

In [1]:
!pip install qdrant-client langchain-huggingface langchain-qdrant langchain-core langchain-openai langchain-community openai transformers sentence-transformers bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are insta

In [2]:
import os
import sys
import json
import gc
import torch
from google.colab import userdata
from qdrant_client import QdrantClient
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_qdrant import QdrantVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.documents import Document
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# sys.path.append(os.path.abspath('..'))

# from config.config import Config
# from rag_pipeline import RAGPipelineBase, RAGBaseline, RAGFusion
import gradio as gr
from dotenv import load_dotenv

# load_dotenv('../.env')

### Config

In [3]:
# load_dotenv()

# class Config:
#     OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
#     HF_TOKEN = os.getenv('HF_TOKEN')
#     QDRANT_URL = os.getenv('QDRANT_URL')
#     QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
#     LANGSMITH_API_KEY = os.getenv('LANGSMITH_API_KEY')
#     LANGSMITH_ENDPOINT = os.getenv('LANGSMITH_ENDPOINT', "https://api.smith.langchain.com")
#     LANGSMITH_PROJECT = os.getenv('LANGSMITH_PROJECT', "skripsi_fixed")
#     EMBEDDING_MODEL = "text-embedding-3-small"
#     EMBEDDING_DIM = 1536
#     BATCH_SIZE = 4
#     MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
#     COLLECTION_NAME = "2026_03_16__1147_enriched_medical_reports"
#     TOP_K = 3
#     TEMPERATURE = 0.1
#     REPETITION_PENALTY = 1.1
#     MAX_NEW_TOKENS = 512
#     PICKLE_PATH = "2026_04_29__1545_medical_documents_enriched.pkl"
#     DATA_FOLDER_PATH = 'knowledge-base/processed-1'
#     RAW_DATA_PATH = 'knowledge-base/raw'
#     PROCESSED_DATA_PATH = 'knowledge-base/processed'
#     GOLDEN_DATASET_PATH = "nutrition_qa_golden_dataset.json"

#     def to_dict(cls):
#         return {k: v for k, v in vars(cls).items() if k.isupper()}

In [ ]:
config = {
    'LANGSMITH_ENDPOINT': "https://api.smith.langchain.com",
    'LANGSMITH_PROJECT': "skripsi",
    'LANGSMITH_API_KEY': userdata.get('LANGSMITH_API_KEY'),
    'OPENAI_API_KEY': userdata.get("OPENAI_API_KEY"),
    'QDRANT_API_KEY': userdata.get("QDRANT_API_KEY"),
    'QDRANT_URL': userdata.get("QDRANT_URL"),
    'HF_TOKEN': userdata.get("HF_TOKEN"),
    'EMBEDDING_MODEL': "intfloat/multilingual-e5-large",
    'COLLECTION_NAME': "2026_05_21__1111_enriched_medical_reports",
    'MODEL_ID': "Qwen/Qwen3-4B-Instruct-2507",
    'TOP_K': 3,
    }

### RAG Pipeline Classes

In [5]:
# class RAGPipelineBase:
#     """Base class to avoid code duplication between Baseline and Fusion."""

#     GEN_PROMPT_TEMPLATE = """<|im_start|>system
# Anda adalah asisten medis profesional. Jawablah pertanyaan klinis berdasarkan KONTEKS yang diberikan.

# ATURAN:
# 1. Jika konteks berasal dari dokumen yang berbeda (sumber berbeda), jawablah dengan hati-hati dan jangan mencampuradukkan data antar pasien.
# 2. Jawablah secara detail namun tetap pada intinya, pastikan semua poin teknis medis yang relevan dari konteks disebutkan.
# 3. Jika informasi pendukung tidak lengkap di konteks, jawablah sesuai informasi yang ada tanpa mengarang fakta medis tambahan. Jika benar-benar tidak ada informasi, nyatakan "Informasi tidak lengkap."
# <|im_end|>
# <|im_start|>user
# KONTEKS:
# {context}

# PERTANYAAN:
# {question}
# <|im_end|>
# <|im_start|>assistant
# """

#     def __init__(self, config, llm=None, vectorstore=None):
#         self.config = config
#         self.setup_environment()

#         if llm:
#             self.llm = llm
#         else:
#             self.setup_llm()

#         if vectorstore:
#             self.vectorstore = vectorstore
#         else:
#             self.setup_vectorstore()

#     def setup_environment(self):
#         os.environ["LANGCHAIN_TRACING_V2"] = "true"
#         os.environ["LANGCHAIN_ENDPOINT"] = self.config.get('LANGSMITH_ENDPOINT', '')
#         os.environ["LANGCHAIN_PROJECT"] = self.config.get('LANGSMITH_PROJECT', '')
#         os.environ["LANGCHAIN_API_KEY"] = self.config.get('LANGSMITH_API_KEY', '')
#         os.environ["OPENAI_API_KEY"] = self.config.get('OPENAI_API_KEY', '')
#         os.environ["HF_TOKEN"] = self.config.get('HF_TOKEN', '')

#     def _build_invoke_config(self, run_name, tags=None, metadata=None):
#         base_tags = ["medical-rag", self.__class__.__name__.lower()]
#         if tags:
#             base_tags.extend(tags)

#     def setup_llm(self):
#         bnb_config = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_compute_dtype=torch.float16,
#         )
#         tokenizer = AutoTokenizer.from_pretrained(self.config['MODEL_ID'])
#         model = AutoModelForCausalLM.from_pretrained(
#             self.config['MODEL_ID'],
#             quantization_config=bnb_config,
#             device_map="auto",
#             trust_remote_code=True
#         )

#         text_pipe = pipeline(
#             "text-generation",
#             model=model,
#             tokenizer=tokenizer,
#             max_new_tokens=512,
#             temperature=0.1,
#             repetition_penalty=1.1,
#             return_full_text=False,
#             stop_sequence=["<|im_end|>", "<|endoftext|>"]
#         )
#         self.llm = HuggingFacePipeline(pipeline=text_pipe)

#     def setup_vectorstore(self):
#         self.client = QdrantClient(url=self.config['QDRANT_URL'], api_key=self.config['QDRANT_API_KEY'], timeout=300)

#         embedding_bnb_config = BitsAndBytesConfig(
#             load_in_4bit=True,
#             bnb_4bit_compute_dtype=torch.float16,
#             bnb_4bit_quant_type="nf4",
#             bnb_4bit_use_double_quant=True,
#         )

#         self.embedding_model = HuggingFaceEmbeddings(
#             model_name=self.config['EMBEDDING_MODEL'],
#             model_kwargs={
#                 "model_kwargs": {"device_map": "cuda", "quantization_config": embedding_bnb_config, "torch_dtype": torch.float16},
#                 "trust_remote_code": True,
#                 "token": self.config["HF_TOKEN"],
#             },
#             encode_kwargs={
#                 "normalize_embeddings": True,
#             })
#         self.vectorstore = QdrantVectorStore(
#             client=self.client,
#             collection_name=self.config['COLLECTION_NAME'],
#             embedding=self.embedding_model,
#             content_payload_key="page_content",
#         )

#     @staticmethod
#     def format_docs(docs):
#         formatted = []
#         for i, doc in enumerate(docs):
#             source = doc.metadata.get('source', 'Unknown')
#             formatted.append(f"--- DOKUMEN {i+1} (Sumber: {source}) ---\n{doc.page_content}")
#         return "\n\n".join(formatted)

#     def retrieve(self, query, k=3, filter_dict=None):
#         instruction = "Representasikan pertanyaan medis ini untuk pencarian rekam medis yang relevan: "
#         if filter_dict:
#             from qdrant_client.http import models as rest
#             must_conditions = [
#                 rest.FieldCondition(key=f"metadata.{key}", match=rest.MatchValue(value=v))
#                 for key, v in filter_dict.items() if v
#             ]
#             qdrant_filter = rest.Filter(must=must_conditions) if must_conditions else None
#             return self.vectorstore.similarity_search(instruction + query, k=k, filter=qdrant_filter)
#         else:
#             return self.vectorstore.similarity_search(instruction + query, k=k)

# class RAGBaseline(RAGPipelineBase):
#     def __init__(self, config, llm=None, vectorstore=None):
#         super().__init__(config, llm, vectorstore)
#         self.setup_generation_chain()

#     def setup_generation_chain(self):
#         self.prompt = ChatPromptTemplate.from_template(self.GEN_PROMPT_TEMPLATE)
#         self.generation_chain = self.prompt | self.llm | StrOutputParser()

#     def run(self, query, filter_dict=None):
#         docs = self.retrieve(query, k=self.config.get('TOP_K', 3), filter_dict=filter_dict)
#         formatted_context = self.format_docs(docs)
#         answer = self.generation_chain.invoke({"context": formatted_context, "question": query})

#         return {
#             "answer": answer,
#             "contexts": docs,
#             "ids": [doc.metadata.get('chunk_id') for doc in docs]
#         }

# class RAGFusion(RAGPipelineBase):
#     def __init__(self, config, llm=None, vectorstore=None):
#         super().__init__(config, llm, vectorstore)
#         self.top_n = self.config.get('TOP_K', 3)
#         self.setup_fusion_pipeline()
#         self.setup_generation_chain()

#     def reciprocal_rank_fusion(self, results, k=60):
#         fused_scores = {}
#         for docs in results:
#             for rank, doc in enumerate(docs):
#                 doc_str = json.dumps({"page_content": doc.page_content, "metadata": doc.metadata})
#                 if doc_str not in fused_scores:
#                     fused_scores[doc_str] = 0
#                 fused_scores[doc_str] += 1 / (rank + k)

#         reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
#         final_docs = []
#         for doc_json, _ in reranked[:self.top_n]:
#             data = json.loads(doc_json)
#             final_docs.append(Document(page_content=data['page_content'], metadata=data['metadata']))
#         return final_docs

#     def setup_fusion_pipeline(self):
#         query_gen_template = """<|im_start|>system
# Anda adalah ahli pencarian rekam medis. Tugas Anda:
# 1. Bedah pertanyaan pengguna menjadi sub-pertanyaan yang lebih sederhana jika kompleks.
# 2. Buat variasi istilah medis (misal: "BBLR" menjadi "Berat Badan Lahir Rendah").
# 3. JANGAN hilangkan informasi penting pengidentifikasi pasien (diagnosis, usia, gender).
# Berikan 3 variasi pertanyaan, satu per baris, tanpa angka.
# <|im_end|>
# <|im_start|>user
# {question}
# <|im_end|>
# <|im_start|>assistant
# """
#         prompt = ChatPromptTemplate.from_template(query_gen_template)
#         self.query_generator = (
#             prompt
#             | ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
#             | StrOutputParser()
#             | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
#         )

#     def setup_generation_chain(self):
#         self.prompt = ChatPromptTemplate.from_template(self.GEN_PROMPT_TEMPLATE)
#         self.generation_chain = self.prompt | self.llm | StrOutputParser()

#     def run(self, query, filter_dict=None):
#         generated_queries = self.query_generator.invoke({"question": query})
#         all_queries = [query] + generated_queries

#         all_results = []
#         for q in all_queries:
#             docs = self.retrieve(q, k=self.top_n * 2, filter_dict=filter_dict)
#             all_results.append(docs)

#         fused_docs = self.reciprocal_rank_fusion(all_results)
#         formatted_context = self.format_docs(fused_docs)
#         answer = self.generation_chain.invoke({
#             "context": formatted_context,
#             "question": query
#         })

#         return {
#             "answer": answer,
#             "contexts": fused_docs,
#             "ids": [doc.metadata.get('chunk_id') for doc in fused_docs]
#         }

In [8]:
class RAGPipelineBase:
    GEN_PROMPT_TEMPLATE = """<|im_start|>system
Anda adalah asisten medis profesional. Jawablah pertanyaan klinis berdasarkan KONTEKS yang diberikan.

ATURAN:
1. Jika konteks berasal dari dokumen yang berbeda (sumber berbeda), jawablah dengan hati-hati dan jangan mencampuradukkan data antar pasien.
2. Jawablah secara detail namun tetap pada intinya, pastikan semua poin teknis medis yang relevan dari konteks disebutkan.
3. Jika informasi pendukung tidak lengkap di konteks, jawablah sesuai informasi yang ada tanpa mengarang fakta medis tambahan. Jika benar-benar tidak ada informasi, nyatakan "Informasi tidak lengkap."
<|im_end|>
<|im_start|>user
KONTEKS:
{context}

PERTANYAAN:
{question}
<|im_end|>
<|im_start|>assistant
"""

    def __init__(self, config=None, llm=None, vectorstore=None):
        self.config = config if config else Config.to_dict()
        self.setup_environment()
        if llm:
            self.llm = llm
        else:
            self.setup_llm()
        if vectorstore:
            self.vectorstore = vectorstore
        else:
            self.setup_vectorstore()

    def setup_environment(self):
        required_keys = [
            'OPENAI_API_KEY',
            'QDRANT_API_KEY',
            'QDRANT_URL',
            'LANGSMITH_API_KEY',
            'LANGSMITH_PROJECT',
            'LANGSMITH_ENDPOINT',
        ]
        missing_keys = [k for k in required_keys if not self.config.get(k)]
        if missing_keys:
            print(f"[Tracing Warning] Missing config keys: {', '.join(missing_keys)}")

        os.environ["LANGCHAIN_TRACING_V2"] = "true"
        os.environ["LANGCHAIN_ENDPOINT"] = self.config.get('LANGSMITH_ENDPOINT', '')
        os.environ["LANGCHAIN_PROJECT"] = self.config.get('LANGSMITH_PROJECT', '')
        os.environ["LANGCHAIN_API_KEY"] = self.config.get('LANGSMITH_API_KEY', '')
        os.environ["OPENAI_API_KEY"] = self.config.get('OPENAI_API_KEY', '')
        os.environ["QDRANT_API_KEY"] = self.config.get('QDRANT_API_KEY', '')
        os.environ["QDRANT_URL"] = self.config.get('QDRANT_URL', '')
        os.environ["HF_TOKEN"] = self.config.get('HF_TOKEN', '')

    def _build_invoke_config(self, run_name, tags=None, metadata=None):
        base_tags = ["medical-rag", self.__class__.__name__.lower()]
        if tags:
            base_tags.extend(tags)

        base_metadata = {
            "collection_name": self.config.get('COLLECTION_NAME'),
            "embedding_model": self.config.get('EMBEDDING_MODEL'),
            "generator_model": self.config.get('MODEL_ID'),
            "top_k": self.config.get('TOP_K', 3),
        }
        if metadata:
            base_metadata.update(metadata)

        return {
            "run_name": run_name,
            "tags": base_tags,
            "metadata": base_metadata,
        }

    def setup_llm(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        tokenizer = AutoTokenizer.from_pretrained(self.config['MODEL_ID'])
        model = AutoModelForCausalLM.from_pretrained(
            self.config['MODEL_ID'],
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        text_pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            temperature=0.1,
            repetition_penalty=1.1,
            return_full_text=True,
            stop_sequence=["<|im_end|>", "<|endoftext|>"]
        )
        self.llm = HuggingFacePipeline(pipeline=text_pipe)

    def setup_vectorstore(self):
        self.client = QdrantClient(url=self.config['QDRANT_URL'], api_key=self.config['QDRANT_API_KEY'], timeout=300)
        try:
            from qdrant_client.http import models
            self.client.create_payload_index(
                collection_name=self.config['COLLECTION_NAME'],
                field_name="metadata.medical_diagnosis",
                field_schema=models.PayloadSchemaType.KEYWORD,
                wait=True
            )
        except Exception as e:
            print(f"Index creation skipped/failed: {e}")
        embedding_bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        self.embedding_model = HuggingFaceEmbeddings(
            model_name=self.config['EMBEDDING_MODEL'],
            model_kwargs={
                "model_kwargs": {"device_map": "cuda", "quantization_config": embedding_bnb_config, "torch_dtype": torch.float16},
                "trust_remote_code": True,
                "token": self.config["HF_TOKEN"],
            },
            encode_kwargs={
                "normalize_embeddings": True,
            })
        self.vectorstore = QdrantVectorStore(
            client=self.client,
            collection_name=self.config['COLLECTION_NAME'],
            embedding=self.embedding_model,
            content_payload_key="page_content",
        )

    @staticmethod
    def format_docs(docs):
        formatted = []
        for i, doc in enumerate(docs):
            source = doc.metadata.get('source', 'Unknown')
            formatted.append(f"--- DOKUMEN {i+1} (Sumber: {source}) ---\n{doc.page_content}")
        return "\n\n".join(formatted)

    def retrieve(self, query, k=3, filter_dict=None, trace_label="vector_retrieval", trace_tags=None, trace_metadata=None):
        instruction = "Representasikan pertanyaan medis ini untuk pencarian rekam medis yang relevan: "
        qdrant_filter = None
        if filter_dict:
            from qdrant_client.http import models as rest
            must_conditions = [
                rest.FieldCondition(key=f"metadata.{key}", match=rest.MatchValue(value=value))
                for key, value in filter_dict.items() if value
            ]
            qdrant_filter = rest.Filter(must=must_conditions) if must_conditions else None

        def _search(_):
            return self.vectorstore.similarity_search(
                instruction + query,
                k=k,
                filter=qdrant_filter,
            )

        retriever = RunnableLambda(_search)
        invoke_config = self._build_invoke_config(
            run_name=trace_label,
            tags=["retrieval"] + (trace_tags or []),
            metadata={
                "query_preview": query[:160],
                "query_length": len(query),
                "filter_used": bool(filter_dict),
                "filter_keys": list(filter_dict.keys()) if filter_dict else [],
                "k": k,
                **(trace_metadata or {}),
            },
        )
        docs = retriever.invoke({}, config=invoke_config)
        return docs


class RAGBaseline(RAGPipelineBase):
    def __init__(self, config=None, llm=None, vectorstore=None):
        super().__init__(config, llm, vectorstore)
        self.setup_generation_chain()

    def setup_generation_chain(self):
        self.prompt = ChatPromptTemplate.from_template(self.GEN_PROMPT_TEMPLATE)
        self.generation_chain = self.prompt | self.llm | StrOutputParser()

    def run(self, query, filter_dict=None):
        docs = self.retrieve(
            query,
            k=self.config.get('TOP_K', 3),
            filter_dict=filter_dict,
            trace_label="baseline_retrieval",
            trace_tags=["baseline"],
            trace_metadata={"method": "baseline"},
        )
        formatted_context = self.format_docs(docs)
        answer = self.generation_chain.invoke(
            {"context": formatted_context, "question": query},
            config=self._build_invoke_config(
                run_name="baseline_generation",
                tags=["baseline", "generation"],
                metadata={
                    "method": "baseline",
                    "retrieved_doc_count": len(docs),
                    "retrieved_chunk_ids": [doc.metadata.get('chunk_id') for doc in docs],
                },
            ),
        )
        return {
            "answer": answer,
            "contexts": docs,
            "ids": [doc.metadata.get('chunk_id') for doc in docs]
        }


class RAGFusion(RAGPipelineBase):
    def __init__(self, config=None, llm=None, vectorstore=None):
        super().__init__(config, llm, vectorstore)
        self.top_n = self.config.get('TOP_K', 3)
        self.setup_fusion_pipeline()
        self.setup_generation_chain()

    def reciprocal_rank_fusion(self, results: list[list], k=60):
        fused_scores = {}
        for docs in results:
            for rank, doc in enumerate(docs):
                try:
                    doc_str = doc.model_dump_json()
                except:
                    doc_str = json.dumps({"page_content": doc.page_content, "metadata": doc.metadata})
                if doc_str not in fused_scores:
                    fused_scores[doc_str] = 0
                fused_scores[doc_str] += 1 / (rank + k)
        reranked = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        final_docs = []
        for doc_json, _ in reranked[:self.top_n]:
            data = json.loads(doc_json)
            final_docs.append(Document(page_content=data['page_content'], metadata=data['metadata']))
        return final_docs

    def setup_fusion_pipeline(self):
        query_gen_template = """<|im_start|>system
Anda adalah ahli pencarian rekam medis. Tugas Anda:
1. Bedah pertanyaan pengguna menjadi sub-pertanyaan yang lebih sederhana jika kompleks.
2. Buat variasi istilah medis (misal: "BBLR" menjadi "Berat Badan Lahir Rendah").
3. JANGAN hilangkan informasi penting pengidentifikasi pasien (diagnosis, usia, gender).
Berikan 3 variasi pertanyaan, satu per baris, tanpa angka.
<|im_end|>
<|im_start|>user
{question}
<|im_end|>
<|im_start|>assistant
"""
        prompt = ChatPromptTemplate.from_template(query_gen_template)
        self.query_generator = (
            prompt
            | ChatOpenAI(model="gpt-4o-mini", temperature=0.2)
            | StrOutputParser()
            | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
        )

    def setup_generation_chain(self):
        self.prompt = ChatPromptTemplate.from_template(self.GEN_PROMPT_TEMPLATE)
        self.generation_chain = self.prompt | self.llm | StrOutputParser()

    def run(self, query, filter_dict=None):
        generated_queries = self.query_generator.invoke(
            {"question": query},
            config=self._build_invoke_config(
                run_name="fusion_query_expansion",
                tags=["fusion", "query-expansion"],
                metadata={"method": "fusion", "original_query_length": len(query)},
            ),
        )
        all_queries = [query] + generated_queries

        all_results = []
        for idx, q in enumerate(all_queries):
            docs = self.retrieve(
                q,
                k=self.top_n * 2,
                filter_dict=filter_dict,
                trace_label=f"fusion_retrieval_q{idx}",
                trace_tags=["fusion"],
                trace_metadata={
                    "method": "fusion",
                    "query_index": idx,
                    "is_original_query": idx == 0,
                },
            )
            all_results.append(docs)

        rrf_chain = RunnableLambda(lambda payload: self.reciprocal_rank_fusion(payload['results']))
        fused_docs = rrf_chain.invoke(
            {"results": all_results},
            config=self._build_invoke_config(
                run_name="fusion_rrf",
                tags=["fusion", "rerank"],
                metadata={
                    "method": "fusion",
                    "candidate_doc_count": sum(len(docs) for docs in all_results),
                    "query_count": len(all_queries),
                },
            ),
        )

        formatted_context = self.format_docs(fused_docs)
        answer = self.generation_chain.invoke(
            {
                "context": formatted_context,
                "question": query
            },
            config=self._build_invoke_config(
                run_name="fusion_generation",
                tags=["fusion", "generation"],
                metadata={
                    "method": "fusion",
                    "generated_query_count": len(generated_queries),
                    "retrieved_doc_count": len(fused_docs),
                    "retrieved_chunk_ids": [doc.metadata.get('chunk_id') for doc in fused_docs],
                },
            ),
        )
        return {
            "answer": answer,
            "contexts": fused_docs,
            "ids": [doc.metadata.get('chunk_id') for doc in fused_docs]
        }

### Inisialisasi Pipeline

In [9]:
print("Initializing shared LLM and VectorStore...")
base = RAGPipelineBase(config=config)

# rag_baseline = RAGBaseline(config=Config.to_dict(), llm=base.llm, vectorstore=base.vectorstore)
# rag_fusion = RAGFusion(config=Config.to_dict(), llm=base.llm, vectorstore=base.vectorstore)
rag_baseline = RAGBaseline(config=config, llm=base.llm, vectorstore=base.vectorstore)
rag_fusion = RAGFusion(config=config, llm=base.llm, vectorstore=base.vectorstore)
print("Initialization complete!")

Initializing shared LLM and VectorStore...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/160k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Initialization complete!


### Fungsi Query + Gradio UI

In [10]:
def ask_question(question, method, diagnosis_filter):
    if not question.strip():
        return [("", "")] * (4 if method == "Both" else 2)

    filter_dict = {"medical_diagnosis": diagnosis_filter.strip()} if diagnosis_filter.strip() else None
    query = f"Diagnosis: {diagnosis_filter.strip()}. Pertanyaan: {question}" if diagnosis_filter.strip() else question

    def format_contexts(docs):
        if not docs:
            return "[Tidak ada dokumen terambil. Periksa filter diagnosis atau koleksi Qdrant.]"

        blocks = []
        for i, d in enumerate(docs, 1):
            metadata_json = json.dumps(d.metadata or {}, ensure_ascii=False, indent=2, default=str)
            content_preview = (d.page_content or "")[:2000]
            blocks.append(
                f"### DOKUMEN {i}\n"
                f"Metadata:\n{metadata_json}\n\n"
                f"Content:\n{content_preview}"
            )

        return "\n\n---\n\n".join(blocks)

    def safe_run(pipeline, query, filter_dict):
        try:
            res = pipeline.run(query, filter_dict=filter_dict)
            return (res['answer'], format_contexts(res['contexts']))
        except Exception as e:
            gc.collect()
            torch.cuda.empty_cache()
            return (f"Error: {e}", "")

    if method == "Baseline RAG":
        ans, ctx = safe_run(rag_baseline, query, filter_dict)
        return [("Baseline RAG", ans), ("Baseline Contexts", ctx)]
    elif method == "RAG-Fusion":
        ans, ctx = safe_run(rag_fusion, query, filter_dict)
        return [("RAG-Fusion", ans), ("Fusion Contexts", ctx)]
    else:
        ans_b, ctx_b = safe_run(rag_baseline, query, filter_dict)
        gc.collect()
        torch.cuda.empty_cache()
        ans_f, ctx_f = safe_run(rag_fusion, query, filter_dict)
        gc.collect()
        torch.cuda.empty_cache()
        return [("Baseline RAG", ans_b), ("Baseline Contexts", ctx_b),
                ("RAG-Fusion", ans_f), ("Fusion Contexts", ctx_f)]


with gr.Blocks(title="Medical RAG - Baseline vs RAG-Fusion") as demo:
    gr.Markdown(
        """
        # Analisis Perbandingan Baseline RAG dan RAG-Fusion pada Dokumen Rekam Medis

        Masukkan pertanyaan klinis seputar data rekam medis (gizi/nutrisi pediatri) untuk membandingkan
        hasil dari **Baseline RAG** (single-vector search) vs **RAG-Fusion** (multi-query + Reciprocal Rank Fusion).
        """
    )

    # --- INPUT SECTION (always visible, fixed width) ---
    with gr.Column(variant="panel"):
        question_input = gr.Textbox(
            label="Pertanyaan Klinis",
            placeholder="Contoh: Bagaimana status gizi pasien berdasarkan data antropometri?",
            lines=3
        )
        method_dropdown = gr.Dropdown(
            choices=["Baseline RAG", "RAG-Fusion", "Both"],
            value="Both",
            label="Metode",
            info="Pilih arsitektur RAG yang akan digunakan"
        )
        diagnosis_input = gr.Textbox(
            label="Filter Diagnosis (opsional)",
            placeholder="Contoh: Gizi Buruk, Gizi Kurang, Stunting",
            info="Batasi pencarian hanya pada diagnosis tertentu"
        )
        submit_btn = gr.Button("Kirim", variant="primary", size="lg")

    # --- OUTPUT SECTION (tabbed so it works on narrow/mobile viewports) ---
    with gr.Tabs():
        with gr.Tab("Baseline RAG"):
            baseline_answer = gr.Textbox(label="Jawaban Baseline RAG", lines=10, buttons=['copy'], show_label=True)
            baseline_context = gr.Textbox(label="Baseline - Retrieved Documents", lines=8, buttons=['copy'], show_label=True)
        with gr.Tab("RAG-Fusion"):
            fusion_answer = gr.Textbox(label="Jawaban RAG-Fusion", lines=10, buttons=['copy'], show_label=True)
            fusion_context = gr.Textbox(label="RAG-Fusion - Retrieved Documents", lines=8, buttons=['copy'], show_label=True)

    def handle_query(question, method, diagnosis):
        outputs = ask_question(question, method, diagnosis)
        ans_b, ctx_b, ans_f, ctx_f = "", "", "", ""
        if method == "Baseline RAG":
            ans_b, ctx_b = outputs[0][1], outputs[1][1]
        elif method == "RAG-Fusion":
            ans_f, ctx_f = outputs[0][1], outputs[1][1]
        else:
            ans_b, ctx_b = outputs[0][1], outputs[1][1]
            ans_f, ctx_f = outputs[2][1], outputs[3][1]
        return (ans_b, ctx_b, ans_f, ctx_f)

    submit_btn.click(
        fn=handle_query,
        inputs=[question_input, method_dropdown, diagnosis_input],
        outputs=[baseline_answer, baseline_context, fusion_answer, fusion_context],
    )


if __name__ == "__main__":
    demo.launch(share=False, theme=gr.themes.Soft())

/tmp/ipykernel_3077/3157078569.py:50: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Medical RAG - Baseline vs RAG-Fusion", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>